# Tokenizer Playground

1. 用 BERT 观察特殊 token：`[UNK]` / `[CLS]` / `[SEP]` / `[PAD]`
2. 对比 Qwen 与 mBERT，并观察 `padding_side`

> BERT 没有名为 BOS/EOS 的 token，对应的是 `[CLS]` / `[SEP]`。

In [ ]:
import os

# 避免 HF Hub / 公司代理把 kernel 卡死；模型已在本地缓存
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", local_files_only=True)

print("special tokens:", tokenizer.special_tokens_map)
print(
    "pad / cls / sep / unk:",
    tokenizer.pad_token,
    tokenizer.cls_token,
    tokenizer.sep_token,
    tokenizer.unk_token,
)

## 1. 特殊 Token 观察

### UNK

词表外字符（如 emoji）会落到 `[UNK]`。

In [ ]:
unk_tokens = tokenizer.tokenize("hello 💩")
print("UNK 示例:", unk_tokens)

### CLS / SEP（对应 BOS / EOS）

- `tokenize()` 只切词，**不加**特殊 token
- `tokenizer()` / `encode()` **默认会加** `[CLS]` 和 `[SEP]`

In [ ]:
print("only tokenize:", tokenizer.tokenize("hello world"))

encoded = tokenizer("hello world")
print("CLS/SEP tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("CLS/SEP ids   :", encoded["input_ids"])

### PAD

batch 里短句补齐到最长句长度时才会出现 `[PAD]`，并用 `attention_mask` 标出有效位。

In [ ]:
batch = tokenizer(
    ["hi", "hello world this is longer"],
    padding=True,
)

print("短句 tokens:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0]))
print("短句 ids   :", batch["input_ids"][0])
print("attention_mask:", batch["attention_mask"][0])
print()
print("长句 tokens:", tokenizer.convert_ids_to_tokens(batch["input_ids"][1]))
print("长句 ids   :", batch["input_ids"][1])
print("attention_mask:", batch["attention_mask"][1])

## 2. Tokenizer 对比实验

- **tokenizer_a**: `Qwen/Qwen2.5-0.5B-Instruct`（BPE）
- **tokenizer_b**: `bert-base-multilingual-cased`（WordPiece）

### 准备

先加载两个 tokenizer，再定义测试文本。

In [ ]:
from transformers import AutoTokenizer

tokenizer_a = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", local_files_only=True
)
tokenizer_b = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", local_files_only=True
)

print("tokenizer_a:", tokenizer_a.name_or_path, "vocab_size=", tokenizer_a.vocab_size)
print("tokenizer_b:", tokenizer_b.name_or_path, "vocab_size=", tokenizer_b.vocab_size)

测试文本：

In [ ]:
texts = [
    "你好，世界！",
    "Transformer inference is interesting.",
    "大模型 inference optimization",
    "KV Cache 可以减少重复计算。",
    "Hello世界",
    "24 GB CUDA GPU",
    "print('hello world')",
    "3.1415926",
    "🙂🚀🔥",
    "  前后有空格  ",
]

### 实验 1：同一句话的 token 数对比

对相同文本分别 tokenize，比较 token 序列与长度差异。

In [ ]:
def inspect_tokenizer(name, tokenizer, text):
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    print(f"{name}:", tokens)
    print(f"{name}_ids:", ids)
    print(f"{name}_length:", len(ids))
    print(f"{name}_decode:", tokenizer.decode(ids))


for item in texts:
    print("original text:", item)
    inspect_tokenizer("tokenizer_a", tokenizer_a, item)
    inspect_tokenizer("tokenizer_b", tokenizer_b, item)
    print("-" * 100)

### 实验 2：`padding_side` left vs right

对同一 batch 分别设置 `padding_side="left"` / `"right"`，打印 `input_ids` 与 `attention_mask`。

**结论：** Padding 改变批处理中的 token 位置和 attention mask，但不应改变原始文本的有效 token。

In [ ]:
batch_texts = ["hi", "hello world this is longer"]


def show_padding(side: str):
    # 用 Qwen 演示；生成式模型推理时常见 left padding
    tokenizer_a.padding_side = side
    batch = tokenizer_a(batch_texts, padding=True)
    print(f"===== padding_side={side!r} =====")
    for i, text in enumerate(batch_texts):
        ids = batch["input_ids"][i]
        mask = batch["attention_mask"][i]
        tokens = tokenizer_a.convert_ids_to_tokens(ids)
        valid_ids = [tid for tid, m in zip(ids, mask) if m == 1]
        print(f"text: {text!r}")
        print("tokens        :", tokens)
        print("input_ids     :", ids)
        print("attention_mask:", mask)
        print("valid_ids     :", valid_ids)
        print("valid_decode  :", tokenizer_a.decode(valid_ids))
        print()


show_padding("right")
show_padding("left")

# 复原，避免影响后续 cell
tokenizer_a.padding_side = "right"

核对：左右 padding 下，`attention_mask == 1` 对应的有效 token 应完全一致。

In [ ]:
# 验证：左右 padding 后，有效 token（mask==1）应一致
right = tokenizer_a(batch_texts, padding=True)
tokenizer_a.padding_side = "left"
left = tokenizer_a(batch_texts, padding=True)
tokenizer_a.padding_side = "right"

for i, text in enumerate(batch_texts):
    right_valid = [tid for tid, m in zip(right["input_ids"][i], right["attention_mask"][i]) if m == 1]
    left_valid = [tid for tid, m in zip(left["input_ids"][i], left["attention_mask"][i]) if m == 1]
    same = right_valid == left_valid
    print(f"{text!r}: valid tokens identical? {same}")
    print("  right valid:", right_valid)
    print("  left  valid:", left_valid)

## 3. Embedding 查表实验

Day 01 任务 3：搞清楚 `nn.Embedding` 到底做了什么。

```text
input_ids [B, S]  →  查 [V, D] 的表  →  hidden_states [B, S, D]
```

### 准备

用极小的 `V=10, D=4`，好让整张表能一眼看完。

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from transformers import GPT2Config

torch.manual_seed(0)  # 固定种子，保证每次运行拿到同一张表

V, D = 10, 4
emb = nn.Embedding(V, D)
ids = torch.tensor([[1, 2, 1], [5, 5, 9]])  # 假装是 tokenizer 的输出

print("emb.weight 形状:", tuple(emb.weight.shape), "← [V, D]，这就是那张表")
print(emb.weight)
print()
print("ids 形状:", tuple(ids.shape), "← [B, S]，dtype:", ids.dtype)
print("查表结果形状:", tuple(emb(ids).shape), "← [B, S, D]")


### 实验 1：查表就是取行

`emb(ids)` 走 `nn.Embedding.forward`；`emb.weight[ids]` 是普通张量的花式索引，完全没经过这一层。

索引的形状规则：用 `[2, 3]` 的整数张量去索引 `[10, 4]` 的第 0 维，得到 `[2, 3] + [4]` = `[2, 3, 4]`。

**结论：** 两条路逐比特相同。Embedding 层没有乘法、没有偏置、没有激活，只是按行号取数。

In [ ]:
lookup = emb(ids)
indexed = emb.weight[ids]

print("emb(ids)        :", tuple(lookup.shape), "grad_fn =", type(lookup.grad_fn).__name__)
print("emb.weight[ids] :", tuple(indexed.shape), "grad_fn =", type(indexed.grad_fn).__name__)
print("逐比特相同:", torch.equal(lookup, indexed))

# ids[0] = [1, 2, 1]，同一个 token 1 出现在位置 0 和位置 2
print()
print("位置 0 (token 1):", [round(x, 4) for x in lookup[0, 0].tolist()])
print("位置 2 (token 1):", [round(x, 4) for x in lookup[0, 2].tolist()])
print("同一 token 不同位置，向量相同:", torch.equal(lookup[0, 0], lookup[0, 2]))

### 实验 2：等价于 one-hot 矩阵乘

教科书常说 embedding 是「one-hot 向量乘权重矩阵」。这里把它真算一遍：`[B, S, V] @ [V, D]` → `[B, S, D]`。

误差为 0 而不是「近似相等」：one-hot 里除一位外全是 0，`0 * x` 精确等于 0，累加 0 也精确，浮点误差没有产生的机会。

**结论：** 数学上等价，工程上完全不可行。换成 GPT-2 尺度就能看出差几个数量级 —— 这就是为什么说 embedding 是**访存密集**而不是计算密集。

In [ ]:
onehot = F.one_hot(ids, num_classes=V)
print("one_hot 形状:", tuple(onehot.shape), "dtype:", onehot.dtype)
print("one_hot[0, 0] (token 1):", onehot[0, 0].tolist(), "← 第 1 位是 1，即行号")

matmul = onehot.float() @ emb.weight  # 必须转 float：权重是 float32
print()
print("矩阵乘:", tuple(onehot.shape), "@", tuple(emb.weight.shape), "->", tuple(matmul.shape))
print("与查表逐比特相同:", torch.equal(matmul, emb(ids)))

# 换成 GPT-2 尺度，对比两条路的代价
B, S, V_GPT2, D_GPT2 = 1, 1024, 50257, 768
print()
print(f"GPT-2 尺度 (B={B}, S={S}, V={V_GPT2}, D={D_GPT2}):")
print(
    f"  one-hot: 中间张量 {B * S * V_GPT2 * 4 / 1e6:.0f} MB,"
    f" 计算量 {2 * B * S * V_GPT2 * D_GPT2 / 1e9:.0f} GFLOP"
)
print(f"  查表   : 只读 {B * S * D_GPT2 * 4 / 1e6:.1f} MB, 计算量 0 FLOP")

### 实验 3：输入形状其实不受限

文档写的是「输入 `(*)` 任意形状，输出 `(*, H)`」—— `nn.Embedding` 只在末尾追加一个长度为 D 的维度。

**结论：** `[B, S] → [B, S, D]` 是本项目自己的约定，不是 PyTorch 的限制。
`TokenEmbedding` 里那句 `ndim != 2` 的检查是主动加的：若误传一维张量，torch 会照常返回结果，
后面 attention 全算错却不报错 —— 这类 bug 极难排查，所以要把「静默算错」变成「立刻报错」。

In [ ]:
for shape in [(3,), (2, 3), (2, 3, 4)]:
    out_shape = emb(torch.zeros(shape, dtype=torch.long)).shape
    print(f"输入 {shape} -> 输出 {tuple(out_shape)}")

print()
print("注意最后一行：输出末尾两个 4 来源不同 —— 前一个来自输入形状，后一个才是 D")
print("（这里 D 恰好也等于 4，纯属巧合；把 D 换成 8 就一眼清楚了）")

emb8 = nn.Embedding(V, 8)
print("D=8 时:", tuple(emb8(torch.zeros((2, 3, 4), dtype=torch.long)).shape))

### 实验 4：padding_idx

指定 `padding_idx` 后，那一行被初始化为全零，且训练时**不接收梯度**，永远保持初始值。
对应的就是 tokenizer 笔记里的 PAD token。

两个容易误解的点：

1. 「全零」只是初始值，可以手动改成别的值，它依然不吃梯度
2. 对纯推理来说必要性不高 —— PAD 位置最终会被 `attention_mask` 屏蔽，embedding 不为零也影响不到有效 token；何况 GPT-2 原生就没有 PAD token

In [ ]:
pad_emb = nn.Embedding(V, D, padding_idx=0)

print("padding_idx =", pad_emb.padding_idx)
print("第 0 行 (PAD) :", pad_emb.weight[0].tolist())
print("第 1 行 (对照):", [round(x, 4) for x in pad_emb.weight[1].tolist()])

### 实验 5：默认初始化与真实尺度

`nn.Embedding` 的权重默认从 N(0, 1) 初始化。

**坑：** GPT-2 的 `initializer_range` 是 **0.02**，和 PyTorch 默认的 1.0 差 50 倍。
加载预训练权重时随机初始值会被整个覆盖，所以不影响；但自己初始化跑前向时若数值炸掉，先怀疑这里。

顺带记住这张表的真实体积：GPT-2 small 一共约 124M 参数，光 token embedding 就占了三成。
再加上权重共享（输出层 `lm_head` 和它是同一个矩阵），这份权重在推理时会被用两次 ——
开头查表是纯访存零计算，结尾算 logits 时转置着做矩阵乘，是全模型最大的 GEMM 之一。

In [ ]:
big = nn.Embedding(50000, 768)
print(f"PyTorch 默认初始化: mean={big.weight.mean():.4f} std={big.weight.std():.4f}")

cfg = GPT2Config()
n_params = cfg.vocab_size * cfg.n_embd
print()
print(f"GPT-2 small: V={cfg.vocab_size}, D={cfg.n_embd}, initializer_range={cfg.initializer_range}")
print(f"  token embedding 参数量: {n_params / 1e6:.1f}M")
print(f"  fp32 {n_params * 4 / 1024 ** 2:.1f} MB / fp16 {n_params * 2 / 1024 ** 2:.1f} MB")

### 小结

| 实验 | 结论 |
|------|------|
| 1 | Embedding = 按行取数，没有任何计算 |
| 2 | 等价于 one-hot 矩阵乘，但代价差几个数量级 → 访存密集，不是计算密集 |
| 3 | `[B, S] → [B, S, D]` 是项目约定，torch 本身不限制输入形状 |
| 4 | `padding_idx` 那一行恒为零、不吃梯度 |
| 5 | 默认 N(0,1)，GPT-2 用 0.02；embedding 表占 GPT-2 small 三成参数 |

**通向 Day 02：** 实验 1 里同一个 token 在位置 0 和位置 2 查到完全相同的向量 ——
Token Embedding 不含位置信息，这正是需要 Positional Embedding 的原因。